> ## SOLUTION / ANSWER KEY &mdash; Lab 7.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-gather-context.ipynb`](../lab-02-gather-context.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.2 &mdash; Gather Context with Tools

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Wrap lookup_order and get_template as @tool functions
- Gather the order + the reply template BEFORE drafting
- See why gather-first prevents hallucinated specifics

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Grounding the task in real data](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-02"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
A general model doesn't know **your** client's order or **your** reply templates &mdash; so the agent
must **gather context first, then draft** (deck slide 6). Gathering happens through **tools** over
your systems: an orders DB, a template store, the CRM. An agent that drafts before it gathers
**hallucinates specifics**; one that grounds every claim in retrieved context is accurate and
auditable.

In [ ]:
from langchain_core.tools import tool

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

print("orders on file :", list(ORDERS))
print("templates on file:", list(TEMPLATES))

## Your Turn
Complete the two gather tools and the `gather` step that pulls both before any drafting.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})

@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind, e.g. delivery_delay or refund."""
    return TEMPLATES.get(kind, "")

def gather(order_id, kind):
    # gather FIRST: pull the order AND the template before we draft anything
    return {"order": lookup_order.invoke(order_id), "template": get_template.invoke(kind)}

try:
    print('order 4471 :', lookup_order.invoke('4471'))
    print('unknown    :', lookup_order.invoke('9999'))
    ctx = gather('4471', 'delivery_delay')
    print('gathered   :', ctx)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("lookup_order finds a real order", lambda: lookup_order.invoke("4471")["status"] == "shipped")
expect_true("an unknown order degrades to 'unknown'", lambda: lookup_order.invoke("9999")["status"] == "unknown")
expect_true("get_template returns the delivery template", lambda: "{name}" in get_template.invoke("delivery_delay"))
expect_true("gather returns BOTH order and template", lambda: set(gather("4471", "delivery_delay")) == {"order", "template"})
expect_true("gather grounds on real data (ETA Friday)", lambda: gather("4471", "delivery_delay")["order"]["eta"] == "Friday")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Gather first, draft second. These are just Module-6 tools pointed at a real job -- the order and the template are the ground truth the draft will stand on.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>